
# QLoRA Fine‑Tuning: `openai/gpt-oss-20b` → FableFlux (Children's Stories)

This notebook fine‑tunes **GPT‑OSS 20B** with **QLoRA (PEFT)** on the dataset:

- Dataset: [`garethpaul/children-stories-dataset`](https://huggingface.co/datasets/garethpaul/children-stories-dataset)
- Base model: [`openai/gpt-oss-20b`](https://huggingface.co/openai/gpt-oss-20b)

It trains a **LoRA adapter** (no full model weights modified), then saves & (optionally) pushes the adapter to the Hugging Face Hub.

> **Notes**
> - GPT‑OSS uses **MXFP4** quantization internally. For training with LoRA we set `Mxfp4Config(dequantize=True)` so modules are usable in bf16.
> - vLLM GPT‑OSS backend currently **does not support LoRA**. To serve with vLLM, later **merge & re‑export to MXFP4** or run via `transformers+peft`.


In [ ]:

# %%capture
!pip install -q transformers==4.56.1 peft==0.12.0 trl==0.9.6 accelerate==0.34.2 datasets==3.0.2 safetensors==0.6.2 bitsandbytes==0.43.3


In [ ]:

import os, json, math, random
from dataclasses import dataclass
from typing import Dict, List

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    Mxfp4Config,
    set_seed,
)
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
set_seed(42)


## (Optional) Login to Hugging Face Hub

In [ ]:

# If pushing adapters to your Hub account, uncomment and run this:
# from huggingface_hub import login
# login(token=os.getenv("HF_TOKEN", ""))  # or paste securely when prompted


In [ ]:

BASE_MODEL = "openai/gpt-oss-20b"
DATASET_ID = "garethpaul/children-stories-dataset"

# Output directories
OUTPUT_DIR = "./gpt-oss-20b-children-qlora"
PUSH_TO_HUB = False                      # set True to push after training
HUB_REPO_ID = "garethpaul/gpt-oss-20b-children-qlora"  # where to push adapter
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Base model:", BASE_MODEL)
print("Dataset:", DATASET_ID)
print("Output dir:", OUTPUT_DIR)


In [ ]:

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
# Ensure a pad token is set; GPT‑OSS often reuses eos as pad
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("pad_token_id:", tokenizer.pad_token_id, "eos_token_id:", tokenizer.eos_token_id)

# Some GPT‑OSS chat templates may be present; we'll use apply_chat_template later.
print("Has chat_template:", tokenizer.chat_template is not None)


## Load dataset & convert to chat messages

In [ ]:

# Expected dataset schema (loose): fields like 'id', 'type', 'characters', 'setting', 'words', 'tags', and story text.
# We'll build a chat-style prompt that asks the model to produce structured JSON stories.

raw = load_dataset(DATASET_ID, split="train")
print(raw[0].keys())

SYSTEM_PROMPT = (
    "You are StoryWeaver, an AI that writes structured children’s bedtime stories.\n"
    "Always respond in valid JSON with keys: {title, characters, setting, story, moral}.\n"
    "The story should be 500–800 words, positive, and end with a moral."
)

def to_messages(example):
    # Try to locate text; fall back to a composed user request using metadata.
    story_text = example.get("text") or example.get("story") or ""
    title = example.get("title") or example.get("id") or "Untitled"
    setting = example.get("setting", "A friendly town at night")
    characters = example.get("characters", ["a brave little car"])
    theme = example.get("type", "bedtime")
    # Build a user content that nudges the model to reproduce structure/style
    user_content = (
        f"Write a bedtime story titled '{title}'. "
        f"Setting: {setting}. Characters: {characters}. Theme: {theme}. "
        f"Keep it gentle and child-friendly."
    )
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ]
    }

ds = raw.shuffle(seed=42).select(range(min(10000, len(raw))))  # cap for example
chat_ds = ds.map(to_messages, remove_columns=ds.column_names)
print(chat_ds[0])


In [ ]:

def format_example(example):
    # Render to text using chat template, then return tokenized later
    rendered = tokenizer.apply_chat_template(
        example["messages"],
        add_generation_prompt=True,
        tokenize=False,
    )
    return {"text": rendered}

text_ds = chat_ds.map(format_example)
print(text_ds[0]["text"][:400])


## Load base model (MXFP4 dequantize=True)

In [ ]:

quant_config = Mxfp4Config(dequantize=True)  # dequantize to train LoRA layers
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.bfloat16,
    quantization_config=quant_config,
    device_map="auto",
    attn_implementation="eager",
)

# Disable cache for gradient checkpointing compatibility
model.config.use_cache = False
print("Model loaded.")


## Configure QLoRA (PEFT) targets for GPT‑OSS MoE

In [ ]:

# GPT‑OSS is MoE; target linear/expert projections.
# Start conservative; you can broaden targets as VRAM allows.
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules="all-linear",   # broad linear targeting
    # You can also explicitly specify expert projections if needed:
    # target_parameters=[
    #     "mlp.experts.gate_up_proj",
    #     "mlp.experts.down_proj",
    # ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


## Train with TRL `SFTTrainer`

In [ ]:

max_seq_len = 2048

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=1,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    lr_scheduler_type="cosine_with_min_lr",
    lr_scheduler_kwargs={"min_lr_rate": 0.1},
    warmup_ratio=0.03,
    logging_steps=10,
    gradient_checkpointing=True,
    bf16=True,
    max_seq_length=max_seq_len,
    packing=False,
    save_strategy="steps",
    save_steps=1000,
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=text_ds,
    tokenizer=tokenizer,
    dataset_text_field="text",
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved LoRA adapter to", OUTPUT_DIR)


## Quick inference (adapter-in-place)

In [ ]:

model.eval()

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Tell me a bedtime story about a brave little car."}
]

prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
enc = tokenizer(prompt, return_tensors="pt", return_attention_mask=True).to(model.device)

with torch.no_grad():
    out = model.generate(
        **enc,
        max_new_tokens=500,
        temperature=0.7,
        top_p=0.9,
    )

print(tokenizer.decode(out[0], skip_special_tokens=True))


## (Optional) Push adapter to Hugging Face Hub

In [ ]:

if PUSH_TO_HUB:
    from huggingface_hub import HfApi
    api = HfApi()
    api.create_repo(HUB_REPO_ID, repo_type="model", exist_ok=True, private=False)
    api.upload_folder(repo_id=HUB_REPO_ID, folder_path=OUTPUT_DIR)
    print("Pushed adapter to:", HUB_REPO_ID)
else:
    print("Set PUSH_TO_HUB=True to upload.")
